# MLP-VaR Softplus SiLU Gated Regime - 2010-2024, w=1

Modelo MLP con rama base y rama de regimen/estres a partir de la familia `Softplus + dynamic SiLU LayerNorm dropout 0.30`.

No se modifica la funcion de perdida, el target ni la definicion de VaR. La perdida sigue siendo exactamente pinball sobre `target` en escala original con `w=1`.

Hipotesis: el fallo de DQ viene de transiciones de regimen. La rama base estima el VaR normal y una rama de estres, activada por un gate aprendido con features ex ante de regimen, permite reaccionar antes a shocks y aceleracion de volatilidad.

In [1]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

torch.set_num_threads(1)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

c:\Users\GONZA\projects\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_LOSS = 1
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

EXPERIMENT_PREFIX = "softplus_silu_gated_regime_2010_2024_w1"
EXPERIMENTS = [
    {
        "key": "silu_gated_regime",
        "label": "Softplus + SiLU Gated Regime",
        "architecture": "silu_gated_regime",
        "feature_set": "full",
        "regime_set": "regime",
        "dropout": 0.30,
        "weight_decay": 0.0,
        "retrain_every": 1,
        "prefix": "softplus_silu_gated_regime_2010_2024_w1",
        "pred_file": "nn_softplus_silu_gated_regime_2010_2024_w1.csv",
    },
]

## Carga de datos y features dinamicas

In [3]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta target"
assert "rp_lag_0" in dataset.columns, "Falta rp_lag_0"
assert "vol_20" in dataset.columns, "Falta vol_20"
assert "vol_60" in dataset.columns, "Falta vol_60"

base_feature_cols = [c for c in dataset.columns if c != "target"]

# Senales del modelo dinamico anterior: ratios de volatilidad, shocks y regimen.
eps = 1e-8
rp = dataset["rp_lag_0"]
abs_r = rp.abs()
downside = np.maximum(-rp, 0.0)
vol_5 = rp.rolling(5).std(ddof=0)
vol_10 = rp.rolling(10).std(ddof=0)
vol_20_realized = rp.rolling(20).std(ddof=0)
vol_20_ref = dataset["vol_20"].abs()
vol_60_ref = dataset["vol_60"].abs()

# Base del mejor MLP anterior.
dataset["vol_5_ratio"] = vol_5 / (vol_20_realized + eps)
dataset["vol_10_ratio"] = vol_10 / (vol_20_realized + eps)
dataset["shock_1"] = abs_r / (vol_20_realized + eps)
dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20_realized + eps)

# Features dinamicas del modelo SiLU anterior.
dataset["vol_20_ratio_60"] = vol_20_ref / (vol_60_ref + eps)
dataset["vol_20_ratio_252"] = vol_20_ref / (vol_20_ref.rolling(252).median() + eps)
dataset["abs_ret_sum_5_ratio"] = abs_r.rolling(5).sum() / (vol_20_realized + eps)
dataset["abs_ret_sum_10_ratio"] = abs_r.rolling(10).sum() / (vol_20_realized + eps)
dataset["max_abs_ret_10_ratio"] = abs_r.rolling(10).max() / (vol_20_realized + eps)

# Features ex ante de transicion de regimen para la rama de estres/gate.
dataset["ewma_abs_3_ratio"] = abs_r.ewm(span=3, adjust=False).mean() / (vol_20_realized + eps)
dataset["ewma_abs_10_ratio"] = abs_r.ewm(span=10, adjust=False).mean() / (vol_20_realized + eps)
dataset["ewma_downside_3_ratio"] = pd.Series(downside, index=dataset.index).ewm(span=3, adjust=False).mean() / (vol_20_realized + eps)
dataset["ewma_downside_10_ratio"] = pd.Series(downside, index=dataset.index).ewm(span=10, adjust=False).mean() / (vol_20_realized + eps)
dataset["downside_cum_5_ratio"] = pd.Series(downside, index=dataset.index).rolling(5).sum() / (vol_20_realized + eps)
dataset["downside_cum_10_ratio"] = pd.Series(downside, index=dataset.index).rolling(10).sum() / (vol_20_realized + eps)
dataset["stress_intensity_5"] = ((pd.Series(downside, index=dataset.index) / (vol_20_realized + eps)) > 1.0).rolling(5).sum()
dataset["stress_intensity_10"] = ((pd.Series(downside, index=dataset.index) / (vol_20_realized + eps)) > 1.0).rolling(10).sum()
dataset["vol_accel_5_20"] = vol_5 / (vol_20_realized + eps)
dataset["vol_accel_10_20"] = vol_10 / (vol_20_realized + eps)

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()

dynamic_features_full = [
    "vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5",
    "vol_20_ratio_60", "vol_20_ratio_252",
    "abs_ret_sum_5_ratio", "abs_ret_sum_10_ratio", "max_abs_ret_10_ratio",
]
regime_features = [
    "ewma_abs_3_ratio", "ewma_abs_10_ratio",
    "ewma_downside_3_ratio", "ewma_downside_10_ratio",
    "downside_cum_5_ratio", "downside_cum_10_ratio",
    "stress_intensity_5", "stress_intensity_10",
    "vol_accel_5_20", "vol_accel_10_20",
]

FEATURE_SETS = {
    "full": base_feature_cols + dynamic_features_full + regime_features,
    "regime": regime_features,
}

print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features full:", len(FEATURE_SETS["full"]))
print("Features regime:", len(FEATURE_SETS["regime"]))
print("Features dinamicas full:", [c for c in dynamic_features_full if c in dataset.columns])
print("Features regime:", [c for c in regime_features if c in dataset.columns])
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

assert len(FEATURE_SETS["full"]) == 41, f"Se esperaban 41 features full, hay {len(FEATURE_SETS['full'])}"
assert len(FEATURE_SETS["regime"]) == 10, f"Se esperaban 10 features regime, hay {len(FEATURE_SETS['regime'])}"
for feature_set in FEATURE_SETS.values():
    assert all(c in dataset.columns for c in feature_set), "Faltan features en dataset"
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece tener escala inesperada"

Dataset: (4680, 42)
Rango: 2006-04-05 -> 2024-12-27
Features full: 41
Features regime: 10
Features dinamicas full: ['vol_5_ratio', 'vol_10_ratio', 'shock_1', 'shock_5', 'vol_20_ratio_60', 'vol_20_ratio_252', 'abs_ret_sum_5_ratio', 'abs_ret_sum_10_ratio', 'max_abs_ret_10_ratio']
Features regime: ['ewma_abs_3_ratio', 'ewma_abs_10_ratio', 'ewma_downside_3_ratio', 'ewma_downside_10_ratio', 'downside_cum_5_ratio', 'downside_cum_10_ratio', 'stress_intensity_5', 'stress_intensity_10', 'vol_accel_5_20', 'vol_accel_10_20']
target mean/std/min/max: -0.00017689210141721946 0.00751750450881661 -0.07361457194412012 0.0784301570334365


## Metricas de backtesting

In [4]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        return {"LRind": LRind, "p_ind": p_ind, "LRcc": np.nan, "p_cc": np.nan}
    LRcc = LRuc + LRind
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": 1 - chi2.cdf(LRcc, df=2)}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    return {"DQ": DQ, "p_dq": 1 - chi2.cdf(DQ, df=X.shape[1])}

In [5]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))


def evaluate_var_model(df, model_label, alpha=0.99, sig=0.05):
    real_losses = df["loss_real"].values
    var_preds = df["VaR_pred"].values
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    violations = int(I.sum())
    violation_rate = violations / T
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    df_2020 = df.loc["2020"] if (df.index.year == 2020).any() else pd.DataFrame(columns=df.columns)
    row = {
        "Model": model_label,
        "w": W_LOSS,
        "T": T,
        "Expected Viol.": (1 - alpha) * T,
        "Violations": violations,
        "Violation Rate": violation_rate,
        "VR Gap": abs(violation_rate - (1 - alpha)),
        "Coverage Bias": violation_rate - (1 - alpha),
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var_preds)),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "Mean VaR 2020": float(df_2020["VaR_pred"].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020["VaR_pred"].max()) if len(df_2020) else np.nan,
        "n_negative_var": int((df["VaR_pred"] < 0).sum()),
        "p_UC": kup["p_uc"],
        "UC Test": "PASS" if kup["p_uc"] > sig else "FAIL",
        "p_CC": cc["p_cc"],
        "CC Test": "PASS" if cc["p_cc"] > sig else "FAIL",
        "p_DQ": dq["p_dq"],
        "DQ Test": "PASS" if dq["p_dq"] > sig else "FAIL",
    }
    vals = [row["p_UC"], row["p_CC"], row["p_DQ"]]
    row["p_Mean"] = float(np.mean([v for v in vals if pd.notnull(v)]))
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return pd.DataFrame([row])


def evaluate_by_year(df, model_label, alpha=0.99):
    rows = []
    for year, g in df.groupby("year"):
        real_losses = g["loss_real"].values
        var_preds = g["VaR_pred"].values
        violations = int(g["viol"].sum())
        expected = (1 - alpha) * len(g)
        kup = kupiec_test(real_losses, var_preds, alpha=alpha)
        cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
        dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
        rows.append({
            "Model": model_label,
            "year": int(year),
            "T": len(g),
            "Expected Viol.": expected,
            "Violations": violations,
            "Violation Rate": violations / len(g),
            "VR Gap": abs((violations / len(g)) - (1 - alpha)),
            "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
            "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
            "Mean VaR Level": float(np.nanmean(var_preds)),
            "Max VaR Level": float(np.nanmax(var_preds)),
            "Min VaR Level": float(np.nanmin(var_preds)),
            "Max Loss": float(np.nanmax(real_losses)),
            "Mean Loss": float(np.nanmean(real_losses)),
            "n_negative_var": int((g["VaR_pred"] < 0).sum()),
            "p_UC": kup["p_uc"],
            "UC Test": "PASS" if kup["p_uc"] > SIG else "FAIL",
            "p_CC": cc["p_cc"],
            "CC Test": "PASS" if cc["p_cc"] > SIG else "FAIL",
            "p_DQ": dq["p_dq"],
            "DQ Test": "PASS" if dq["p_dq"] > SIG else "FAIL",
        })
    return pd.DataFrame(rows)


def violation_table(df):
    cols = ["VaR_pred", "loss_real", "viol", "year"]
    out = df.loc[df["viol"] == 1, cols].copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["exceedance_ratio"] = out["loss_real"] / out["VaR_pred"]
    return out.sort_index()


def worst_days_table(df, n=25):
    out = df.copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["loss_minus_var"] = out["exceedance"]
    return out.sort_values("loss_real", ascending=False).head(n)


def violation_gap_summary(df):
    v = df.loc[df["viol"] == 1].copy()
    if len(v) < 2:
        return {"violations": len(v), "median_gap_days": np.nan, "gaps_le_5": 0, "gaps_le_20": 0}
    gaps = pd.Series(v.index).diff().dt.days.dropna()
    return {
        "violations": len(v),
        "median_gap_days": float(gaps.median()),
        "gaps_le_5": int((gaps <= 5).sum()),
        "gaps_le_20": int((gaps <= 20).sum()),
    }

## Modelos y entrenamiento

In [6]:
def inverse_softplus(y):
    return math.log(math.expm1(y))


# IMPORTANTE: no modificar. Misma perdida pinball que los experimentos Softplus anteriores.
def var_loss(y_true, y_pred, q=0.99, w=1.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplusSiLUGatedRegime(nn.Module):
    def __init__(self, d_in, d_regime, dropout=0.30):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(dropout),
        )
        self.stress = nn.Sequential(
            nn.Linear(d_regime, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(d_regime, 32),
            nn.SiLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        self.out = nn.Linear(64, 1)
        self.softplus = nn.Softplus()
        nn.init.constant_(self.out.bias, inverse_softplus(0.02))

    def forward(self, x, x_regime):
        h_base = self.base(x)
        h_stress = self.stress(x_regime)
        gate = self.gate(x_regime)
        h = (1.0 - gate) * h_base + gate * h_stress
        return self.softplus(self.out(h))


def build_model(d_in, d_regime, architecture, dropout):
    if architecture == "silu_gated_regime":
        return MLPVaRSoftplusSiLUGatedRegime(d_in=d_in, d_regime=d_regime, dropout=dropout)
    raise ValueError(f"Arquitectura no soportada: {architecture}")


def train_one_model(
    X_train,
    X_regime_train,
    y_train,
    d_in,
    d_regime,
    seed,
    w_loss,
    architecture,
    dropout,
    weight_decay,
    alpha=0.99,
    max_epochs=200,
    lr=5e-5,
    batch_size=256,
    patience=15,
    val_split=0.10,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    R_tr, R_val = X_regime_train[:split], X_regime_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]
    model = build_model(d_in=d_in, d_regime=d_regime, architecture=architecture, dropout=dropout)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(R_tr), torch.from_numpy(y_tr)),
        batch_size=batch_size,
        shuffle=False,
    )
    X_val_t = torch.from_numpy(X_val)
    R_val_t = torch.from_numpy(R_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for xb, rb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb, rb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t, R_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Ejecucion de experimentos

In [7]:
def run_experiment(exp):
    eval_dates = dataset.loc[pd.Timestamp(EVAL_START):pd.Timestamp(EVAL_END)].index
    feature_cols = FEATURE_SETS[exp["feature_set"]]
    regime_cols = FEATURE_SETS[exp["regime_set"]]
    regime_idx = [feature_cols.index(c) for c in regime_cols]
    retrain_every = exp.get("retrain_every", RETRAIN_EVERY)
    var_preds = []
    real_targets = []
    dates = []
    current_model = None
    muX, sdX = None, None

    desc = f"{exp['key']} w=1"
    for i, current_date in enumerate(tqdm(eval_dates, desc=desc, dynamic_ncols=True)):
        if i % retrain_every == 0:
            train_end = current_date - pd.Timedelta(days=1)
            train_df = dataset.loc[:train_end].tail(WINDOW)
            if len(train_df) < 250:
                if current_model is None:
                    continue
            else:
                X_train = train_df[feature_cols].values.astype(np.float32)
                y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
                muX = X_train.mean(axis=0, keepdims=True)
                sdX = X_train.std(axis=0, keepdims=True) + 1e-8
                X_train = (X_train - muX) / sdX
                X_regime_train = X_train[:, regime_idx]
                current_model = train_one_model(
                    X_train,
                    X_regime_train,
                    y_train,
                    d_in=X_train.shape[1],
                    d_regime=X_regime_train.shape[1],
                    seed=SEED,
                    w_loss=W_LOSS,
                    architecture=exp["architecture"],
                    dropout=exp["dropout"],
                    weight_decay=exp["weight_decay"],
                    alpha=ALPHA,
                )

        if current_model is None or muX is None or sdX is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
        X_regime_test = X_test[:, regime_idx]
        y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

        current_model.eval()
        with torch.no_grad():
            pred = current_model(torch.from_numpy(X_test), torch.from_numpy(X_regime_test)).numpy().reshape(-1)[0]

        var_preds.append(float(pred))
        real_targets.append(float(y_test.reshape(-1)[0]))
        dates.append(current_date)

    pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
    pred_df = pred_df.loc[EVAL_START:EVAL_END].dropna().copy()
    pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
    pred_df["year"] = pred_df.index.year
    return pred_df


results = {}
summary_parts = []
yearly_parts = []
gap_parts = []

for exp in EXPERIMENTS:
    print(f"\n{'=' * 90}\nEjecutando: {exp['label']}\n{'=' * 90}")
    pred_df = run_experiment(exp)
    results[exp["key"]] = pred_df

    summary = evaluate_var_model(pred_df, model_label=exp["label"], alpha=ALPHA, sig=SIG)
    yearly = evaluate_by_year(pred_df, model_label=exp["label"], alpha=ALPHA)
    violations = violation_table(pred_df)
    gaps = violation_gap_summary(pred_df)
    gaps["Model"] = exp["label"]

    pred_path = DATA_DIR / exp["pred_file"]
    summary_path = DATA_DIR / f"{exp['prefix']}_summary.csv"
    yearly_path = DATA_DIR / f"{exp['prefix']}_yearly.csv"
    violations_path = DATA_DIR / f"{exp['prefix']}_violations.csv"

    pred_df.to_csv(pred_path)
    summary.to_csv(summary_path, index=False)
    yearly.to_csv(yearly_path, index=False)
    violations.to_csv(violations_path)

    print("Guardado:", pred_path)
    print("Guardado:", summary_path)
    print("Guardado:", yearly_path)
    print("Guardado:", violations_path)
    print("Config:", {k: exp[k] for k in ["dropout", "weight_decay", "retrain_every"]})
    plausibility_report(pred_df)
    display(summary)
    display(yearly)
    display(violations)

    summary_parts.append(summary)
    yearly_parts.append(yearly)
    gap_parts.append(gaps)

new_summary = pd.concat(summary_parts, ignore_index=True)
new_yearly = pd.concat(yearly_parts, ignore_index=True)
gap_summary = pd.DataFrame(gap_parts)


Ejecutando: Softplus + SiLU Gated Regime


silu_gated_regime w=1: 100%|██████████| 3762/3762 [17:41<00:00,  3.54it/s]

Guardado: ..\data\nn_softplus_silu_gated_regime_2010_2024_w1.csv
Guardado: ..\data\softplus_silu_gated_regime_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_silu_gated_regime_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_silu_gated_regime_2010_2024_w1_violations.csv
Config: {'dropout': 0.3, 'weight_decay': 0.0, 'retrain_every': 1}

loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.010014193132519722
max: 0.02912864275276661
mean: 0.017554454753004066
p95: 0.02280992120504379
p99: 0.02504223188385366
p99.9: 0.027307981381192867
n_negative_var: 0
max VaR: 2021-12-20 00:00:00 0.02912864275276661
min VaR: 2024-07-31 00:00:00 0.010014193132519722


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + SiLU Gated Regime,1,3762,37.62,48,0.012759,0.002759,0.002759,0.000284,0.01777,0.017554,0.029129,0.010014,0.017489,0.026082,0,0.102848,PASS,0.095306,PASS,2.220446e-16,FAIL,0.066051,NO


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + SiLU Gated Regime,2010,252,2.52,4,0.015873,0.005873,0.000241,0.017893,0.017772,0.025139,0.011529,0.026197,-0.000381,0,3.880381e-01,PASS,0.645764,PASS,0.972465,PASS
1,Softplus + SiLU Gated Regime,2011,252,2.52,5,0.019841,0.009841,0.000282,0.018179,0.017978,0.027406,0.010869,0.026702,-0.000277,0,1.662402e-01,PASS,0.346492,PASS,0.069153,PASS
2,Softplus + SiLU Gated Regime,2012,249,2.49,3,0.012048,0.002048,0.000181,0.017582,0.017573,0.028943,0.011145,0.019407,-0.000117,0,7.529928e-01,PASS,0.917363,PASS,0.998349,PASS
3,Softplus + SiLU Gated Regime,2013,251,2.51,2,0.007968,0.002032,0.000248,0.017403,0.017250,0.025490,0.011831,0.029816,0.000066,0,7.373115e-01,PASS,0.930176,PASS,0.999324,PASS
4,Softplus + SiLU Gated Regime,2014,252,2.52,1,0.003968,0.006032,0.000190,0.017024,0.016975,0.025172,0.010721,0.025232,0.000438,0,2.731770e-01,PASS,0.546423,PASS,0.818540,PASS
5,Softplus + SiLU Gated Regime,2015,252,2.52,6,0.023810,0.013810,0.000232,0.017408,0.017278,0.025532,0.010107,0.019679,0.000456,0,6.141418e-02,PASS,0.150117,PASS,0.733481,PASS
6,Softplus + SiLU Gated Regime,2016,250,2.50,3,0.012000,0.002000,0.000192,0.017837,0.017817,0.025457,0.010627,0.016855,-0.000410,0,7.579883e-01,PASS,0.919379,PASS,0.998415,PASS
7,Softplus + SiLU Gated Regime,2017,249,2.49,0,0.000000,0.010000,0.000180,0.017498,0.017498,0.024197,0.011423,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + SiLU Gated Regime,2018,250,2.50,2,0.008000,0.002000,0.000187,0.017435,0.017403,0.025927,0.010605,0.018189,0.000311,0,7.419327e-01,PASS,0.932010,PASS,0.000000,FAIL
9,Softplus + SiLU Gated Regime,2019,251,2.51,0,0.000000,0.010000,0.000180,0.017416,0.017416,0.026418,0.010445,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL


,VaR_pred,loss_real,viol,year,exceedance,exceedance_ratio
2010-02-03,0.016139,0.026197,1,2010,0.010058,1.623191
2010-05-03,0.015020,0.017654,1,2010,0.002634,1.175358
2010-05-13,0.012900,0.013289,1,2010,0.000388,1.030103
2010-10-18,0.015092,0.017095,1,2010,0.002003,1.132697
2011-05-04,0.017458,0.023495,1,2011,0.006037,1.345831
2011-08-17,0.013923,0.016823,1,2011,0.002900,1.208274
2011-09-21,0.016578,0.026702,1,2011,0.010124,1.610676
2011-09-27,0.013975,0.015219,1,2011,0.001244,1.089026
2011-12-13,0.018988,0.023733,1,2011,0.004745,1.249894
2012-06-20,0.018968,0.019407,1,2012,0.000439,1.023140


## Comparacion final

In [8]:
def load_prediction_csv(path, var_col="VaR_pred"):
    df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    df = df.loc[EVAL_START:EVAL_END].copy()
    if var_col != "VaR_pred":
        df = df.rename(columns={var_col: "VaR_pred"})
    if "viol" not in df.columns:
        df["viol"] = (df["loss_real"] > df["VaR_pred"]).astype(int)
    if "year" not in df.columns:
        df["year"] = df.index.year
    return df

comparison_rows = []
gap_rows = []

reference_files = [
    ("Softplus base w=1", DATA_DIR / "nn_softplus_validation_w1.csv", DATA_DIR / "nn_softplus_test_w1.csv"),
    ("Softplus + ratios + shocks", DATA_DIR / "nn_softplus_pruebas_shocks_2010_2024_w1.csv", None),
    ("Softplus + dynamic SiLU LayerNorm dropout 0.20", DATA_DIR / "nn_softplus_silu_dropout020_2010_2024_w1.csv", None),
    ("Softplus + dynamic SiLU LayerNorm dropout 0.30", DATA_DIR / "nn_softplus_silu_dropout030_2010_2024_w1.csv", None),
    ("Softplus + dynamic SiLU LayerNorm dropout 0.30 ensemble 3 seeds", DATA_DIR / "nn_softplus_silu_dropout030_ensemble3_2010_2024_w1.csv", None),
]

for label, path_a, path_b in reference_files:
    if not path_a.exists():
        continue
    if path_b is not None and path_b.exists():
        ref_df = pd.concat([load_prediction_csv(path_a), load_prediction_csv(path_b)]).sort_index()
        ref_df = ref_df[~ref_df.index.duplicated(keep="last")].copy()
        ref_df = ref_df.loc[EVAL_START:EVAL_END]
    else:
        ref_df = load_prediction_csv(path_a)
    comparison_rows.append(evaluate_var_model(ref_df, model_label=label, alpha=ALPHA, sig=SIG))
    gaps = violation_gap_summary(ref_df)
    gaps["Model"] = label
    gap_rows.append(gaps)

comparison_rows.append(new_summary)
gap_rows.extend(gap_parts)

comparison = pd.concat(comparison_rows, ignore_index=True)
comparison_path = DATA_DIR / f"{EXPERIMENT_PREFIX}_comparison.csv"
comparison.to_csv(comparison_path, index=False)

gap_comparison = pd.DataFrame(gap_rows)
gap_path = DATA_DIR / f"{EXPERIMENT_PREFIX}_gap_comparison.csv"
gap_comparison.to_csv(gap_path, index=False)

print("Guardado:", comparison_path)
print("Guardado:", gap_path)
print("\nConfirmacion metodologica: la funcion var_loss es la misma pinball que en los experimentos Softplus previos.")

cols = [
    "Model", "w", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
    "Mean VaR 2020", "Max VaR 2020", "n_negative_var", "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
]

display(comparison[cols].sort_values(["DQ Test", "CC Test", "UC Test", "p_DQ", "VR Gap"], ascending=[False, False, False, False, True]))

print("\nResumen de gaps entre violaciones")
display(gap_comparison[["Model", "violations", "median_gap_days", "gaps_le_5", "gaps_le_20"]])

print("\nComparacion anual de los nuevos experimentos")
display(new_yearly.sort_values(["Model", "year"]))

print("\nAnos problematicos de los nuevos experimentos")
display(new_yearly.sort_values(["Violation Rate", "Violations"], ascending=False).head(30))

Guardado: ..\data\softplus_silu_gated_regime_2010_2024_w1_comparison.csv
Guardado: ..\data\softplus_silu_gated_regime_2010_2024_w1_gap_comparison.csv

Confirmacion metodologica: la funcion var_loss es la misma pinball que en los experimentos Softplus previos.


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
3,Softplus + dynamic SiLU LayerNorm dropout 0.30,1,3762,37.62,44,0.011696,0.001696,0.001696,0.000303,0.022319,0.022159,0.071054,0.006583,0.027358,0.068162,0,0.308644,PASS,0.495335,PASS,1.059871e-02,FAIL,0.271526,NO
2,Softplus + dynamic SiLU LayerNorm dropout 0.20,1,3762,37.62,43,0.011430,0.001430,0.001430,0.000302,0.022331,0.022172,0.070854,0.006568,0.027371,0.068118,0,0.388732,PASS,0.560315,PASS,8.034862e-03,FAIL,0.319027,NO
5,Softplus + SiLU Gated Regime,1,3762,37.62,48,0.012759,0.002759,0.002759,0.000284,0.017770,0.017554,0.029129,0.010014,0.017489,0.026082,0,0.102848,PASS,0.095306,PASS,2.220446e-16,FAIL,0.066051,NO
1,Softplus + ratios + shocks,1,3762,37.62,32,0.008506,0.001494,-0.001494,0.000274,0.018858,0.018684,0.028376,0.008985,0.019036,0.028376,0,0.344590,PASS,0.060605,PASS,0.000000e+00,FAIL,0.135065,NO
4,Softplus + dynamic SiLU LayerNorm dropout 0.30...,1,3762,37.62,26,0.006911,0.003089,-0.003089,0.000282,0.021614,0.021482,0.047653,0.009961,0.024146,0.043948,0,0.043771,FAIL,0.052073,PASS,1.626477e-13,FAIL,0.031948,NO
0,Softplus base w=1,1,3762,37.62,36,0.009569,0.000431,-0.000431,0.000283,0.018417,0.018217,0.030089,0.007403,0.017326,0.030089,0,0.789181,PASS,0.016972,FAIL,0.000000e+00,FAIL,0.268718,NO



Resumen de gaps entre violaciones


,Model,violations,median_gap_days,gaps_le_5,gaps_le_20
0,Softplus base w=1,36,35.0,8,16
1,Softplus + ratios + shocks,32,53.0,8,12
2,Softplus + dynamic SiLU LayerNorm dropout 0.20,43,67.0,5,11
3,Softplus + dynamic SiLU LayerNorm dropout 0.30,44,66.0,5,12
4,Softplus + dynamic SiLU LayerNorm dropout 0.30...,26,89.0,5,8
5,Softplus + SiLU Gated Regime,48,49.0,9,21



Comparacion anual de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + SiLU Gated Regime,2010,252,2.52,4,0.015873,0.005873,0.000241,0.017893,0.017772,0.025139,0.011529,0.026197,-0.000381,0,3.880381e-01,PASS,0.645764,PASS,0.972465,PASS
1,Softplus + SiLU Gated Regime,2011,252,2.52,5,0.019841,0.009841,0.000282,0.018179,0.017978,0.027406,0.010869,0.026702,-0.000277,0,1.662402e-01,PASS,0.346492,PASS,0.069153,PASS
2,Softplus + SiLU Gated Regime,2012,249,2.49,3,0.012048,0.002048,0.000181,0.017582,0.017573,0.028943,0.011145,0.019407,-0.000117,0,7.529928e-01,PASS,0.917363,PASS,0.998349,PASS
3,Softplus + SiLU Gated Regime,2013,251,2.51,2,0.007968,0.002032,0.000248,0.017403,0.017250,0.025490,0.011831,0.029816,0.000066,0,7.373115e-01,PASS,0.930176,PASS,0.999324,PASS
4,Softplus + SiLU Gated Regime,2014,252,2.52,1,0.003968,0.006032,0.000190,0.017024,0.016975,0.025172,0.010721,0.025232,0.000438,0,2.731770e-01,PASS,0.546423,PASS,0.818540,PASS
5,Softplus + SiLU Gated Regime,2015,252,2.52,6,0.023810,0.013810,0.000232,0.017408,0.017278,0.025532,0.010107,0.019679,0.000456,0,6.141418e-02,PASS,0.150117,PASS,0.733481,PASS
6,Softplus + SiLU Gated Regime,2016,250,2.50,3,0.012000,0.002000,0.000192,0.017837,0.017817,0.025457,0.010627,0.016855,-0.000410,0,7.579883e-01,PASS,0.919379,PASS,0.998415,PASS
7,Softplus + SiLU Gated Regime,2017,249,2.49,0,0.000000,0.010000,0.000180,0.017498,0.017498,0.024197,0.011423,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + SiLU Gated Regime,2018,250,2.50,2,0.008000,0.002000,0.000187,0.017435,0.017403,0.025927,0.010605,0.018189,0.000311,0,7.419327e-01,PASS,0.932010,PASS,0.000000,FAIL
9,Softplus + SiLU Gated Regime,2019,251,2.51,0,0.000000,0.010000,0.000180,0.017416,0.017416,0.026418,0.010445,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL



Anos problematicos de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
10,Softplus + SiLU Gated Regime,2020,251,2.51,14,0.055777,0.045777,0.001272,0.019691,0.017489,0.026082,0.010724,0.078430,-0.000757,0,4.018639e-07,FAIL,0.000001,FAIL,0.000001,FAIL
5,Softplus + SiLU Gated Regime,2015,252,2.52,6,0.023810,0.013810,0.000232,0.017408,0.017278,0.025532,0.010107,0.019679,0.000456,0,6.141418e-02,PASS,0.150117,PASS,0.733481,PASS
12,Softplus + SiLU Gated Regime,2022,251,2.51,5,0.019920,0.009920,0.000274,0.018017,0.017818,0.028396,0.011424,0.026797,0.000322,0,1.640397e-01,PASS,0.342892,PASS,0.070676,PASS
1,Softplus + SiLU Gated Regime,2011,252,2.52,5,0.019841,0.009841,0.000282,0.018179,0.017978,0.027406,0.010869,0.026702,-0.000277,0,1.662402e-01,PASS,0.346492,PASS,0.069153,PASS
0,Softplus + SiLU Gated Regime,2010,252,2.52,4,0.015873,0.005873,0.000241,0.017893,0.017772,0.025139,0.011529,0.026197,-0.000381,0,3.880381e-01,PASS,0.645764,PASS,0.972465,PASS
2,Softplus + SiLU Gated Regime,2012,249,2.49,3,0.012048,0.002048,0.000181,0.017582,0.017573,0.028943,0.011145,0.019407,-0.000117,0,7.529928e-01,PASS,0.917363,PASS,0.998349,PASS
6,Softplus + SiLU Gated Regime,2016,250,2.50,3,0.012000,0.002000,0.000192,0.017837,0.017817,0.025457,0.010627,0.016855,-0.000410,0,7.579883e-01,PASS,0.919379,PASS,0.998415,PASS
8,Softplus + SiLU Gated Regime,2018,250,2.50,2,0.008000,0.002000,0.000187,0.017435,0.017403,0.025927,0.010605,0.018189,0.000311,0,7.419327e-01,PASS,0.932010,PASS,0.000000,FAIL
3,Softplus + SiLU Gated Regime,2013,251,2.51,2,0.007968,0.002032,0.000248,0.017403,0.017250,0.025490,0.011831,0.029816,0.000066,0,7.373115e-01,PASS,0.930176,PASS,0.999324,PASS
11,Softplus + SiLU Gated Regime,2021,252,2.52,2,0.007937,0.002063,0.000224,0.017562,0.017470,0.029129,0.011090,0.030524,-0.000423,0,7.327118e-01,PASS,0.928317,PASS,0.000000,FAIL


## Lectura esperada

Un experimento solo es candidato si mejora el fallo del modelo 2 sin destruir la cobertura global:

- objetivo principal: `UC`, `CC` y `DQ` globales en `PASS`;
- si `DQ` no pasa, mirar si sube mucho `p_DQ` frente al modelo 2;
- la tasa de violacion deberia quedar cerca del 1%;
- el VaR medio no debe subir de forma generalizada;
- marzo de 2020 y 2022 son los anos clave para diagnosticar clustering.